# ⚡ VectorForge AI Sprite & Vector Lab
### Isolated Prototyping Playground for Diffusion Models, Prompts, Alpha Knockout & Vectorization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/somerandomguy-coder/VectorForge/blob/main/VectorForge_Sprite_Diffusion_Playground.ipynb)

This notebook provides a sandboxed environment to experiment with:
1. **Diffusion Models:** Test ultra-fast 1-step models (`sd-turbo`, `sdxl-turbo`) and checkpoint/pixel-art models (`stable-diffusion-v1-5`, etc.).
2. **Prompt Engineering:** Tailor prompts for clean game sprites, pixel art, RPG items, flat icons, and vector logos.
3. **Alpha Knockout & Background Removal:** Test luminance thresholding (matching VectorForge's client-side engine), auto chroma-keying, and AI background removal (`rembg`).
4. **Vectorization (Raster to SVG):** Trace raster outputs into scalable SVG paths using `vtracer` and Potrace-style contour tracing.
5. **Sprite Utilities:** Downsample to true pixel art (palette quantization) and slice sprite sheets.
6. **Interactive GUI:** An inline `ipywidgets` dashboard to adjust seeds, styles, and steps with 1-click downloads before integrating into [VectorForge](https://github.com/somerandomguy-coder/VectorForge).

---

## 1. Environment Setup & Dependency Installation
Run this cell in Google Colab to install `diffusers`, `transformers`, `torch`, `vtracer`, `rembg`, and visual dependencies.

In [ ]:
# Install core ML and imaging libraries
!pip install -q diffusers transformers accelerate safetensors
!pip install -q vtracer rembg svgwrite matplotlib ipywidgets pillow

import os
import io
import sys
import time
import torch
import numpy as np
from PIL import Image, ImageOps, ImageFilter
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output, SVG
import ipywidgets as widgets

# Check hardware acceleration
device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None (CPU mode)"
print(f"🚀 PyTorch Version: {torch.__version__}")
print(f"🎮 Active Device: {device.upper()} ({gpu_name})")
if device == "cpu":
    print("⚠️ Tip: In Colab, select Runtime > Change runtime type > GPU (T4 or A100) for 10x faster generation!")

## 2. Model Selection & Pipeline Loader
Load distilled diffusion models. For rapid 1-step sprite prototyping, `stabilityai/sd-turbo` runs in ~0.5–1s on a T4 GPU.

In [ ]:
from diffusers import AutoPipelineForText2Image, DiffusionPipeline, DPMSolverMultistepScheduler

AVAILABLE_MODELS = {
    "SD-Turbo (Ultra-Fast 1-Step)": "stabilityai/sd-turbo",
    "SDXL-Turbo (High-Res 1-Step)": "stabilityai/sdxl-turbo",
    "SD 1.5 (Classic / LoRA-compatible)": "runwayml/stable-diffusion-v1-5",
}

active_pipeline = None
current_model_id = None

def load_diffusion_pipeline(model_key="SD-Turbo (Ultra-Fast 1-Step)"):
    global active_pipeline, current_model_id
    model_id = AVAILABLE_MODELS.get(model_key, model_key)
    
    if active_pipeline is not None and current_model_id == model_id:
        print(f"✅ Pipeline '{model_id}' is already loaded.")
        return active_pipeline
        
    print(f"⏳ Loading model '{model_id}' onto {device}...")
    start_t = time.time()
    
    dtype = torch.float16 if device == "cuda" else torch.float32
    
    pipe = AutoPipelineForText2Image.from_pretrained(
        model_id,
        torch_dtype=dtype,
        variant="fp16" if device == "cuda" else None
    )
    
    pipe = pipe.to(device)
    
    if device == "cuda":
        pipe.enable_attention_slicing()
        
    active_pipeline = pipe
    current_model_id = model_id
    print(f"🎉 Loaded in {time.time() - start_t:.2f} seconds!")
    return active_pipeline

# Pre-load default pipeline
pipeline = load_diffusion_pipeline("SD-Turbo (Ultra-Fast 1-Step)")

## 3. Sprite & Vector Prompt Presets
Clean sprite generation requires strict prompt framing: ensuring the asset is isolated on a solid background (white or black), avoiding photorealistic gradients, and suppressing drop shadows.

In [ ]:
PROMPT_STYLES = {
    "Minimal Vector Logo": {
        "prefix": "minimalist flat vector logo emblem of",
        "suffix": "clean 2D silhouette, sharp edges, solid pure white background, corporate identity mark, high contrast, centered",
        "negative": "photorealistic, 3d render, complex shading, gradients, text, watermark, messy lines, dirty background",
        "steps": 1,
        "guidance": 0.0,  # 0.0 recommended for SD-Turbo
    },
    "Modern App Icon": {
        "prefix": "modern rounded app icon of",
        "suffix": "sleek glyph, flat design, Apple iOS design aesthetic, solid white background, high contrast, centered",
        "negative": "photograph, noisy, messy, realistic textures, words, typography, complex background",
        "steps": 1,
        "guidance": 0.0,
    },
    "16-Bit RPG Sprite / Item": {
        "prefix": "16-bit retro game sprite of",
        "suffix": "pixel art asset, clean black outline, vibrant colors, isolated on solid pure white background, centered, SNES RPG style",
        "negative": "blur, realistic photo, gradients, anti-aliased shadows, messy scene, text, cropped",
        "steps": 2,
        "guidance": 0.5,
    },
    "Isometric Game Prop": {
        "prefix": "isometric game asset of",
        "suffix": "low-poly clean vector prop, vibrant colors, isolated on solid white background, 45-degree angle, sharp silhouette",
        "negative": "complex terrain, background scenery, noise, realistic photograph, lens flare",
        "steps": 2,
        "guidance": 0.5,
    },
    "Pixel Character Sprite Sheet": {
        "prefix": "pixel art character sprite sheet of",
        "suffix": "4 action poses, retro arcade hero, clean isolated solid white background, grid arrangement, crisp pixels",
        "negative": "photo, 3d, continuous gradient, blurry, cut off edges, watermark",
        "steps": 2,
        "guidance": 0.5,
    },
    "Geometric Badge / Crest": {
        "prefix": "geometric badge logo of",
        "suffix": "hexagonal border, symmetrical vector linework, clean stamp, solid white background, centered",
        "negative": "busy background, gradient, drop shadow, photograph, blur",
        "steps": 1,
        "guidance": 0.0,
    },
    "Chibi Game Sticker": {
        "prefix": "cute chibi character sticker of",
        "suffix": "bold white die-cut contour outline, solid black background, cute kawaii game mascot, flat vector colors",
        "negative": "photorealistic, messy, noise, human skin pores, watermark",
        "steps": 2,
        "guidance": 0.5,
    }
}

def format_prompt(keyword: str, style_name: str):
    style = PROMPT_STYLES.get(style_name, PROMPT_STYLES["Minimal Vector Logo"])
    full_prompt = f"{style['prefix']} {keyword.strip()}, {style['suffix']}"
    return full_prompt, style["negative"], style["steps"], style["guidance"]

# Preview prompt construction
p, neg, st, gd = format_prompt("cybernetic sword with neon glow", "16-Bit RPG Sprite / Item")
print("✨ Sample Built Prompt:", p)
print("🚫 Negative Prompt:", neg)

## 4. Diffusion Generation Engine
Run inference with reproducible seeds and multi-seed grid exploration.

In [ ]:
def generate_single_sprite(
    keyword: str,
    style_name: str = "Minimal Vector Logo",
    seed: int = 42,
    steps: int = None,
    guidance_scale: float = None,
    width: int = 512,
    height: int = 512
) -> Image.Image:
    """Generates a single raster image using the configured pipeline."""
    prompt, negative_prompt, default_steps, default_guidance = format_prompt(keyword, style_name)
    num_steps = steps if steps is not None else default_steps
    guidance = guidance_scale if guidance_scale is not None else default_guidance
    
    generator = torch.Generator(device=device).manual_seed(seed)
    
    kwargs = {
        "prompt": prompt,
        "num_inference_steps": max(1, num_steps),
        "guidance_scale": guidance,
        "generator": generator,
        "width": width,
        "height": height,
    }
    
    # Only pass negative prompt if model supports classifier-free guidance (guidance > 0)
    if guidance > 0:
        kwargs["negative_prompt"] = negative_prompt
        
    output = active_pipeline(**kwargs)
    return output.images[0]

def generate_seed_grid(keyword: str, style_name: str, seeds=[42, 123, 777, 999]):
    """Runs batch generation across multiple seeds to compare stylistic variations."""
    print(f"🎨 Generating 4 seeds for '{keyword}' ({style_name})...")
    images = [generate_single_sprite(keyword, style_name, seed=s) for s in seeds]
    
    fig, axes = plt.subplots(1, len(seeds), figsize=(16, 4))
    for ax, img, s in zip(axes, images, seeds):
        ax.imshow(img)
        ax.set_title(f"Seed {s}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return images

# Quick test
sample_img = generate_single_sprite("geometric golden eagle crest", "Minimal Vector Logo", seed=42)
display(sample_img.resize((256, 256)))

## 5. Alpha Knockout & Background Removal Engine
Compare 3 background removal approaches:
1. **VectorForge Luminance Thresholding:** Fast, runs client-side in browser OffscreenCanvas.
2. **Chroma / Corner Keying:** Automatically detects background color from image corners.
3. **AI `rembg` (U2-Net):** Deep learning segmentation for anti-aliased and complex edges.

In [ ]:
def luminance_knockout(image: Image.Image, threshold: int = 210, invert: bool = False) -> Image.Image:
    """
    Replicates VectorForge's client-side alpha knockout (alpha-knockout.ts).
    Luminance = 0.299*R + 0.587*G + 0.114*B.
    """
    rgba = image.convert("RGBA")
    arr = np.array(rgba, dtype=np.uint8)
    
    r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]
    luminance = 0.299 * r + 0.587 * g + 0.114 * b
    
    if invert:
        mask = luminance < threshold  # solid black knockout
    else:
        mask = luminance > threshold  # solid white knockout
        
    # Set alpha channel to 0 for background pixels
    arr[:, :, 3] = np.where(mask, 0, 255)
    return Image.fromarray(arr, mode="RGBA")

def chroma_corner_knockout(image: Image.Image, tolerance: int = 35) -> Image.Image:
    """
    Samples the 4 corners of the image to deduce the background color,
    then masks all pixels within Euclidean color distance < tolerance.
    """
    rgba = image.convert("RGBA")
    arr = np.array(rgba, dtype=np.float32)
    h, w, _ = arr.shape
    
    # Sample corners (top-left, top-right, bottom-left, bottom-right)
    corners = np.array([
        arr[0:5, 0:5, :3].mean(axis=(0, 1)),
        arr[0:5, w-5:w, :3].mean(axis=(0, 1)),
        arr[h-5:h, 0:5, :3].mean(axis=(0, 1)),
        arr[h-5:h, w-5:w, :3].mean(axis=(0, 1)),
    ])
    bg_color = np.median(corners, axis=0)
    
    # Color distance
    diff = np.linalg.norm(arr[:, :, :3] - bg_color, axis=-1)
    mask = diff < tolerance
    
    out_arr = np.array(rgba)
    out_arr[:, :, 3] = np.where(mask, 0, 255)
    return Image.fromarray(out_arr, mode="RGBA")

def ai_rembg_knockout(image: Image.Image) -> Image.Image:
    """AI-powered background removal using rembg (U2-Net)."""
    import rembg
    return rembg.remove(image)

def create_checkerboard(width, height, cell_size=16):
    """Creates a checkerboard background pattern to visually inspect alpha transparency."""
    c1 = (220, 220, 220)
    c2 = (255, 255, 255)
    board = Image.new("RGB", (width, height), c1)
    for y in range(0, height, cell_size):
        for x in range(0, width, cell_size):
            if (x // cell_size + y // cell_size) % 2 == 0:
                for dy in range(min(cell_size, height - y)):
                    for dx in range(min(cell_size, width - x)):
                        board.putpixel((x + dx, y + dy), c2)
    return board

def preview_knockout_comparison(raw_img: Image.Image):
    """Plots raw image against all 3 knockout techniques over a checkerboard."""
    k1 = luminance_knockout(raw_img, threshold=210)
    k2 = chroma_corner_knockout(raw_img, tolerance=35)
    k3 = ai_rembg_knockout(raw_img)
    
    w, h = raw_img.size
    chk = create_checkerboard(w, h)
    
    def overlay_on_chk(img_rgba):
        bg = chk.copy()
        bg.paste(img_rgba, (0, 0), img_rgba)
        return bg
        
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    axes[0].imshow(raw_img)
    axes[0].set_title("1. Raw Diffusion Output")
    axes[0].axis("off")
    
    axes[1].imshow(overlay_on_chk(k1))
    axes[1].set_title("2. Luminance Knockout (VectorForge)")
    axes[1].axis("off")
    
    axes[2].imshow(overlay_on_chk(k2))
    axes[2].set_title("3. Corner Chroma-Key")
    axes[2].axis("off")
    
    axes[3].imshow(overlay_on_chk(k3))
    axes[3].set_title("4. AI RemBG (Deep Learning)")
    axes[3].axis("off")
    
    plt.tight_layout()
    plt.show()
    return k1, k2, k3

_ = preview_knockout_comparison(sample_img)

## 6. Vectorization Engine (Raster to Scalable SVG)
Convert raster masks into clean SVG vectors:
- **Color Vectorization:** Uses `vtracer` to produce multi-color vector paths.
- **Monochrome Silhouette Mark:** Converts to single-color vector mark with instant recoloring (matching VectorForge's Potrace pipeline).

In [ ]:
import vtracer
import re

def raster_to_svg_vtracer(
    image: Image.Image,
    colormode: str = "color",  # 'color' or 'binary'
    hierarchical: str = "stacked",
    filter_speckle: int = 4,
    color_precision: int = 6,
    corner_threshold: int = 60
) -> str:
    """
    Traces an image into SVG XML using VTracer.
    """
    # Save temp png buffer
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    raw_bytes = buf.getvalue()
    
    svg_str = vtracer.convert_raw_image_to_svg(
        raw_bytes,
        img_format="png",
        colormode=colormode,
        hierarchical=hierarchical,
        filter_speckle=filter_speckle,
        color_precision=color_precision,
        corner_threshold=corner_threshold
    )
    return svg_str

def recolor_svg_paths(svg_str: str, new_color_hex: str) -> str:
    """
    Recolors all paths inside SVG string (mirrors vectorizer.ts recolorSvg).
    """
    recolored = re.sub(r'fill="[^"]*"', f'fill="{new_color_hex}"', svg_str)
    if 'fill="' not in recolored:
        recolored = re.sub(r'<path\s+', f'<path fill="{new_color_hex}" ', recolored)
    return recolored

def display_svg(svg_content: str, width: int = 300, height: int = 300):
    """Displays raw SVG string directly in notebook cell."""
    # Inject width and height for proper inline rendering
    styled_svg = re.sub(r'<svg', f'<svg width="{width}" height="{height}"', svg_content, count=1)
    display(HTML(f"<div style='background:#1e1e2e; padding:12px; border-radius:8px; display:inline-block;'>{styled_svg}</div>"))

# Test vectorization on our masked sample
masked = luminance_knockout(sample_img, threshold=210)
sample_svg = raster_to_svg_vtracer(masked, colormode="color")
print(f"📐 Generated SVG Length: {len(sample_svg):,} characters")
display_svg(sample_svg, width=256, height=256)

## 7. Sprite Utilities: Pixel Art Snapper & Sheet Slicer
Tools specifically for game development workflows:
1. **Pixel Art Downsampler:** Snaps smooth diffusion outputs into pixel grids with quantized retro color palettes (e.g. 16 or 32 colors).
2. **Sprite Sheet Slicer:** Splits a 2x2 or 4x4 generation grid into isolated transparent frames.

In [ ]:
def to_retro_pixel_art(
    image: Image.Image,
    target_resolution: int = 48,
    max_colors: int = 16
) -> Image.Image:
    """
    Converts an image to crisp retro pixel art via downsampling and color quantization.
    """
    orig_size = image.size
    # 1. Downscale with BILINEAR or BOX
    small = image.resize((target_resolution, target_resolution), Image.Resampling.BILINEAR)
    
    # 2. Quantize colors
    if small.mode == 'RGBA':
        alpha = small.split()[3]
        rgb = small.convert('RGB').quantize(colors=max_colors, method=Image.Quantize.MEDIANCUT).convert('RGB')
        rgb.putalpha(alpha)
        quantized = rgb
    else:
        quantized = small.quantize(colors=max_colors).convert('RGB')
        
    # 3. Upscale back using NEAREST NEIGHBOR to preserve hard pixel edges
    pixel_art = quantized.resize(orig_size, Image.Resampling.NEAREST)
    return pixel_art

def slice_sprite_sheet(sheet_img: Image.Image, rows: int = 2, cols: int = 2) -> list:
    """
    Slices a generated sprite sheet grid into individual frame images.
    """
    w, h = sheet_img.size
    cell_w = w // cols
    cell_h = h // rows
    frames = []
    
    for r in range(rows):
        for c in range(cols):
            box = (c * cell_w, r * cell_h, (c + 1) * cell_w, (r + 1) * cell_h)
            frames.append(sheet_img.crop(box))
            
    return frames

# Quick demonstration of pixel art conversion
pixelated_sample = to_retro_pixel_art(sample_img, target_resolution=32, max_colors=16)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(sample_img)
ax1.set_title("Original Diffusion")
ax1.axis("off")
ax2.imshow(pixelated_sample)
ax2.set_title("Snappy 32x32 Pixel Art (16 Colors)")
ax2.axis("off")
plt.tight_layout()
plt.show()

## 8. Interactive Prototyping Dashboard
Use the interactive controls below to experiment with keywords, styles, seeds, and download your finished assets.

In [ ]:
# State holders
last_generated_raw = None
last_masked_png = None
last_svg_str = None

# UI Widgets
txt_keyword = widgets.Text(
    value="cybernetic fox emblem with circuit traces",
    placeholder="Enter sprite subject or keywords...",
    description="Keyword:",
    layout=widgets.Layout(width="70%")
)

dropdown_style = widgets.Dropdown(
    options=list(PROMPT_STYLES.keys()),
    value="Minimal Vector Logo",
    description="Style:",
    layout=widgets.Layout(width="45%")
)

slider_seed = widgets.IntSlider(value=42, min=1, max=99999, step=1, description="Seed:")
slider_steps = widgets.IntSlider(value=1, min=1, max=8, step=1, description="Steps:")
slider_guidance = widgets.FloatSlider(value=0.0, min=0.0, max=4.0, step=0.1, description="Guidance:")

dropdown_knockout = widgets.Dropdown(
    options=["VectorForge Luminance (White BG)", "VectorForge Luminance Invert (Black BG)", "Corner Chroma-Key", "AI RemBG"],
    value="VectorForge Luminance (White BG)",
    description="Knockout:",
    layout=widgets.Layout(width="45%")
)

slider_threshold = widgets.IntSlider(value=210, min=50, max=255, step=1, description="Threshold:")

btn_generate = widgets.Button(
    description="⚡ Generate & Trace Asset",
    button_style="primary",
    icon="bolt",
    layout=widgets.Layout(width="280px", height="42px")
)

btn_download_png = widgets.Button(description="💾 Download Transparent PNG", icon="image", button_style="success")
btn_download_svg = widgets.Button(description="📐 Download Scalable SVG", icon="code", button_style="info")

output_area = widgets.Output()

def on_style_change(change):
    style = PROMPT_STYLES.get(change.new, {})
    slider_steps.value = style.get("steps", 1)
    slider_guidance.value = style.get("guidance", 0.0)

dropdown_style.observe(on_style_change, names="value")

def run_generation_flow(_):
    global last_generated_raw, last_masked_png, last_svg_str
    with output_area:
        clear_output(wait=True)
        print(f"⏳ Generating '{txt_keyword.value}' with seed {slider_seed.value}...")
        
        raw = generate_single_sprite(
            keyword=txt_keyword.value,
            style_name=dropdown_style.value,
            seed=slider_seed.value,
            steps=slider_steps.value,
            guidance_scale=slider_guidance.value
        )
        last_generated_raw = raw
        
        # Knockout
        ko_mode = dropdown_knockout.value
        if ko_mode == "VectorForge Luminance (White BG)":
            masked = luminance_knockout(raw, threshold=slider_threshold.value, invert=False)
        elif ko_mode == "VectorForge Luminance Invert (Black BG)":
            masked = luminance_knockout(raw, threshold=slider_threshold.value, invert=True)
        elif ko_mode == "Corner Chroma-Key":
            masked = chroma_corner_knockout(raw, tolerance=35)
        else:
            masked = ai_rembg_knockout(raw)
        last_masked_png = masked
        
        # Vectorize
        svg_code = raster_to_svg_vtracer(masked, colormode="color")
        last_svg_str = svg_code
        
        # Render 3-Stage Visual Preview
        clear_output()
        w, h = raw.size
        chk = create_checkerboard(w, h)
        bg = chk.copy()
        bg.paste(masked, (0, 0), masked)
        
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(raw)
        axes[0].set_title("1. Raw Diffusion Output")
        axes[0].axis("off")
        
        axes[1].imshow(bg)
        axes[1].set_title("2. Masked Alpha Asset")
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()
        
        print(f"3. Vectorized SVG Mark ({len(svg_code):,} bytes):")
        display_svg(svg_code, width=280, height=280)

def download_png_click(_):
    if last_masked_png is None:
        print("Generate an image first!")
        return
    filename = f"{txt_keyword.value[:20].strip().replace(' ', '_')}_sprite.png"
    last_masked_png.save(filename)
    try:
        from google.colab import files
        files.download(filename)
    except Exception:
        print(f"✅ Saved locally as: {filename}")

def download_svg_click(_):
    if last_svg_str is None:
        print("Generate an image first!")
        return
    filename = f"{txt_keyword.value[:20].strip().replace(' ', '_')}_vector.svg"
    with open(filename, "w") as f:
        f.write(last_svg_str)
    try:
        from google.colab import files
        files.download(filename)
    except Exception:
        print(f"✅ Saved locally as: {filename}")

btn_generate.on_click(run_generation_flow)
btn_download_png.on_click(download_png_click)
btn_download_svg.on_click(download_svg_click)

# Display Dashboard
controls_box = widgets.VBox([
    txt_keyword,
    widgets.HBox([dropdown_style, slider_seed]),
    widgets.HBox([slider_steps, slider_guidance]),
    widgets.HBox([dropdown_knockout, slider_threshold]),
    widgets.HBox([btn_generate, btn_download_png, btn_download_svg]),
])

display(controls_box)
display(output_area)

## 9. Bridging Back to VectorForge
Once you identify the best prompt framing, model weights, or sprite parameters in this isolated Colab notebook, here is how you bring them into the VectorForge codebase:

### A. Adding New Presets to VectorForge
Open [`src/ui/style-presets.ts`](file:///src/ui/style-presets.ts) in VectorForge and add your tested prompt keywords to `STYLE_PRESETS`:
```typescript
export const STYLE_PRESETS = {
  'rpg-sprite': {
    name: '16-Bit RPG Sprite',
    icon: '🗡️',
    description: 'Retro game item with sharp outline',
    prefix: '16-bit retro game sprite of',
    suffix: 'clean silhouette, vibrant colors, solid white background, centered',
    negativePrompt: 'blurry, 3d, realistic photo, gradients, watermark'
  },
  // ...
};
```

### B. Exporting Model Weights to ONNX for WebGPU Browser Execution
VectorForge runs ONNX models inside the browser using `onnxruntime-web` with the WebGPU provider. To export your PyTorch diffusion model to ONNX, run:
```bash
!pip install optimum[exporters] onnx
!optimum-cli export onnx --model stabilityai/sd-turbo --task text-to-image ./sd-turbo-onnx/
```
The resulting UNet and VAE `.onnx` files can then be served statically or streamed into VectorForge's OPFS manager ([`src/workers/storage-manager.ts`](file:///src/workers/storage-manager.ts)).